# LLM-Culture on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flowersteam/LLM-Culture/blob/dev/notebooks/llm_culture_colab.ipynb)

[LLM-Culture](https://github.com/flowersteam/LLM-Culture) studies the **cultural evolution of text in populations of LLMs**: agents are wired into a network, each reads its neighbours' stories and rewrites them according to a personality and a transformation rule, and you watch the stories evolve over generations.

This notebook runs a full experiment in the cloud. The flow is: **edit one `CONFIG` dict → run the simulation → analyze → (optionally) compare several conditions.** You don't need to know Python beyond editing the dict.

**Tip:** for a real (~7B) model, pick a GPU runtime first — **Runtime → Change runtime type → GPU (T4 is fine)**. The default `CONFIG` uses a tiny model that runs even without a GPU.

New to LLMs? See [docs/llm_for_researchers.md](https://github.com/flowersteam/LLM-Culture/blob/dev/docs/llm_for_researchers.md) for a plain-language primer (model, backend, context window, quantization).

## Clone the repository

In [ ]:
!git clone -b dev https://github.com/flowersteam/LLM-Culture.git

## Install the required packages

The project is managed with [uv](https://docs.astral.sh/uv/). We use it only to write out the exact dependency list, then install those into Colab's Python so the imports below work directly (we add the repo to the path rather than installing it as a package). This takes a couple of minutes the first time.

`llama-cpp-python` is the backend for running a small model locally (the default). For a large model on a GPU runtime, also run the optional vLLM cell further down.

In [ ]:
%cd /content/LLM-Culture
!pip install -q uv
!uv pip install --system -q -e . llama-cpp-python

### Imports

In [ ]:
import os, json
import networkx as nx
import matplotlib.pyplot as plt
%matplotlib inline

from llm_culture.simulation.utils import (
    register_entry,
    register_custom_network_structure,
    build_network_structure,
    run_experiment,
)
from scripts.run_analysis import main_analysis
from scripts.run_comparison_analysis import run_comparison_analysis
from llm_culture.analysis.plots import display_graph

print('Imports OK. Working directory:', os.getcwd())

## Define the experiment by editing the `CONFIG` dict

Everything about a run lives in this one dict. The fields:

- **`n_agents` / `n_timesteps` / `n_seeds`** — population size, number of generations, and how many independent repeats of the whole run.
- **`network_structure`** — who reads whom: `fully_connected`, `sequence` (a chain), `circle`, `caveman` (sub-groups joined by a bridge), or a custom one (see below). `n_cliques` is only used for `caveman`.
- **`prompt_init`** — the task that seeds the very first story.
- **`prompt_update`** — the rule each agent applies to its neighbours' stories every generation (copy, recombine, innovate...).
- **`personalities`** — one entry per agent (list length = `n_agents`); each is a content bias prepended to the prompts. Use the same one for everyone, or mix them.
- **`model` / `llm_backend`** — which model and how it runs: `llama.cpp` (local, CPU/GPU) or `vllm` (Linux GPU). For a remote OpenAI-compatible server set `llm_backend` to anything else and fill `access_url`.
- **`output_name`** — results are written under `results/experiments/<output_name>/`.
- **`debug`** — `True` prints extra detail about each model request (verbose logging).

In [ ]:
CONFIG = {
    # --- population / topology ---
    'n_agents': 5,
    'n_timesteps': 3,
    'n_seeds': 1,
    'network_structure': 'fully_connected',   # 'sequence' | 'circle' | 'caveman' | 'fully_connected' | a custom name
    'n_cliques': 2,                            # only used when network_structure == 'caveman'

    'prompt_init': {
        'name': 'kid',
        'prompt': 'Imagine that you are telling a story to your kid. What would that story be? '
                  'Just output the story, nothing else.',
    },
    'prompt_update': {
        'name': 'Combine2',
        'prompt': 'Combine the stories you have received into a single story. Just output the story, nothing else.',
    },

    # one entry per agent (length must equal n_agents); reuse the same one for all agents if you like
    'personalities': [{'name': 'empty', 'prompt': ''}] * 5,

    # which plots the analysis step should produce
    'plots': [
        'plot_similarity_matrix',
        'plot_between_gen_similarities',
        'plot_word_chains',
        'plot_similarity_graph',
        'plot_init_generation_similarity_evolution',
        'plot_within_gen_similarities',
        'plot_successive_generations_similarities',
    ],

    # --- run settings ---
    'output_name': 'test_experiment',          # results saved under results/experiments/<output_name>/
    'debug': False,                            # True = print extra request detail (verbose logging)
    'llm_backend': 'llama.cpp',                # 'llama.cpp' (local) | 'vllm' (Linux GPU)
    'model': 'unsloth/SmolLM2-135M-Instruct-GGUF',   # HF repo id or local path; tiny default model
    'access_url': None,                        # set to a server URL for a remote OpenAI-compatible backend
    # optional: 'temperature': 0.8,            # copying fidelity (low) vs. innovation (high)
}

## (Optional) Custom network structures

Beyond the built-in topologies you can define any graph with [networkx](https://networkx.org/) and register it, then point `CONFIG['network_structure']` at it. Run **one** of the two cells below (skip this section entirely to use the built-in `network_structure` set above).

### Option 1 — define the edges manually

In [ ]:
n_agents = CONFIG['n_agents']

custom_graph = nx.Graph()
custom_graph.add_nodes_from(range(n_agents))
custom_graph.add_edges_from([(0, 1), (0, 2), (1, 3), (1, 4), (2, 5)])  # example edges — edit freely

register_custom_network_structure('custom_graph', custom_graph)
CONFIG['network_structure'] = 'custom_graph'
display_graph(custom_graph)

### Option 2 — use a networkx graph generator

In [ ]:
n_agents = CONFIG['n_agents']
custom_graph = nx.star_graph(n_agents - 1)  # a star: node 0 at the centre, everyone else on a spoke

register_custom_network_structure('custom_graph', custom_graph)
CONFIG['network_structure'] = 'custom_graph'
display_graph(custom_graph)

## Run the experiment

This simulates all seeds and writes the stories to `results/experiments/<output_name>/`. The first run downloads the model (cached afterwards).

In [ ]:
results_folder = run_experiment(CONFIG)
print('Results in:', results_folder)

## Launch the analysis

Computes the similarity metrics and renders the plots listed in `CONFIG['plots']` (shown inline and saved into the results folder).

In [ ]:
font_sizes = {'ticks': 12, 'labels': 14, 'title': 16}

main_analysis(results_folder, font_sizes=font_sizes, plot=True, plot_configs=CONFIG.get('plots'))

## Compare independent experiments

Define several conditions, run each, then plot them side by side. The example below contrasts a fully-creative population, a fully-uncreative one, and a half-and-half mix.

In [ ]:
import copy

BASE_CONFIG = {
    'n_agents': 2,
    'n_timesteps': 3,
    'n_seeds': 2,
    'network_structure': 'fully_connected',
    'n_cliques': 2,
    'prompt_init': {
        'name': 'kid',
        'prompt': 'Imagine that you are telling a story to your kid. What would that story be? '
                  'Just output the story, nothing else.',
    },
    'prompt_update': {
        'name': 'Combine2',
        'prompt': 'Combine the stories you have received into a single story. Just output the story, nothing else.',
    },
    'personalities': [{'name': 'empty', 'prompt': ''}] * 2,
    'plots': [
        'plot_similarity_matrix',
        'plot_between_gen_similarities',
        'plot_word_chains',
        'plot_similarity_graph',
        'plot_init_generation_similarity_evolution',
        'plot_within_gen_similarities',
        'plot_successive_generations_similarities',
    ],
    'output_name': 'test_experiment',
    'debug': False,
    'llm_backend': 'llama.cpp',
    'model': 'unsloth/SmolLM2-135M-Instruct-GGUF',
    'access_url': None,
}

# Three variants that differ only in personality.
CREATIVE_CONFIG = copy.deepcopy(BASE_CONFIG)
CREATIVE_CONFIG['personalities'] = [{'name': 'Creative', 'prompt': 'For what follows, pretend that you are a very creative person.'}] * CREATIVE_CONFIG['n_agents']
CREATIVE_CONFIG['output_name'] = 'test_creative_agents'

NOT_CREATIVE_CONFIG = copy.deepcopy(BASE_CONFIG)
NOT_CREATIVE_CONFIG['personalities'] = [{'name': 'NotCreative', 'prompt': 'For what follows, pretend that you are a very uncreative person.'}] * NOT_CREATIVE_CONFIG['n_agents']
NOT_CREATIVE_CONFIG['output_name'] = 'test_not_creative_agents'

MIXED_CONFIG = copy.deepcopy(BASE_CONFIG)
MIXED_CONFIG['personalities'] = (
    [{'name': 'Creative', 'prompt': 'For what follows, pretend that you are a very creative person.'}] * (MIXED_CONFIG['n_agents'] // 2)
    + [{'name': 'NotCreative', 'prompt': 'For what follows, pretend that you are a very uncreative person.'}] * (MIXED_CONFIG['n_agents'] // 2)
)
MIXED_CONFIG['output_name'] = 'test_mixed_agents'

variants = [
    ('CREATIVE', CREATIVE_CONFIG),
    ('NOT_CREATIVE', NOT_CREATIVE_CONFIG),
    ('MIXED', MIXED_CONFIG),
]

In [ ]:
variant_folders = []
variant_labels = []

for label, cfg in variants:
    folder = run_experiment(cfg)
    variant_folders.append(folder)
    variant_labels.append(label)

print(variant_folders)

In [ ]:
comparison_sizes = {'ticks': 16, 'labels': 18, 'legend': 16, 'title': 23, 'matrix': 8}

comparison_plot_configs = [
    'compare_similarity_matrix',
    'compare_init_generation_similarity_evolution',
    'compare_within_generation_similarity_evolution',
    'compare_successive_generations_similarities',
]

run_comparison_analysis(
    variant_folders,
    plot=True,
    scale_y_axis=False,
    labels=variant_labels,
    sizes=comparison_sizes,
    plot_configs=comparison_plot_configs,
)

## (Optional) Run a large model on a GPU with vLLM

On a **GPU runtime**, install vLLM and switch the backend to run a real ~7B model. (vLLM is Linux/GPU-only; skip this on a CPU runtime and keep `llama.cpp`.)

In [ ]:
!uv pip install --system -q vllm

CONFIG['llm_backend'] = 'vllm'
CONFIG['model'] = 'mistralai/Mistral-7B-Instruct-v0.2'
# then re-run the 'Run the experiment' + 'Launch the analysis' cells above

## Using the web interface

Experiments can also be launched from a code-free web interface. The cell below starts a local server, but the printed URL will **not** open from inside Colab (the server runs on the Colab VM, not your machine). To use the GUI, download this notebook and run it locally, then open the printed URL (typically `http://127.0.0.1:5002`).

1. Open the simulation page and enter an experiment name (results go to `results/experiments/<name>/`).
2. Set the number of agents, generations, and seeds.
3. Choose a network structure (`fully_connected`, `sequence`, `circle`, `caveman`; for `caveman` also set the number of cliques).
4. Pick the initial prompt, update prompt, and personality for each agent from the registered lists — or use the "Add Prompt" form to register a new one first.
5. Choose the LLM mode: a remote OpenAI-compatible server URL, or a local path / Hugging Face repo id.
6. Start the run; the page shows progress and redirects to the analysis when it finishes.
7. Use the analysis / comparison pages to generate plots.

In [ ]:
!python web_interface.py

## Saving your results

Colab storage is temporary — everything under `results/` is lost when the VM is recycled. To keep outputs, mount your Google Drive and copy them across:

```python
from google.colab import drive
drive.mount('/content/drive')
!cp -r results /content/drive/MyDrive/llm_culture_results
```